# 第 37 课｜MultiIndex、分组与行业中性化

本 Notebook 对应「365 天量化金融」第 37 课。课程代码主线来自本地视频、课件截图和 Whisper 口播整理；在此基础上补充量化数据语义、断言、性能验证和方法边界。

- 图文口播：`/Users/mojiawen/Documents/学习资料归档/_暂存/pam-video-captures/365-days-quant-finance/37_youtube-95k-gTun3mg/index.html`
- 环境：`pkm-quant`
- 数据：固定随机种子生成的离线模拟行情（**非真实股票数据**）
- 状态：可运行学习稿；不代表策略收益承诺

> **数据声明**：本 Notebook 的证券名称、价格和成交量均为教学模拟数据。它用于稳定复现计算方法，不能用于评价真实证券、市场规律或策略收益。

## 本节量化问题

一篮子股票看似来自多个行业，并不代表资金真正分散。本节用 pandas 面板数据回答四个问题：

1. 如何在“日期 × 股票”宽表与带双层索引的长表之间转换；
2. 如何按行业执行 split–apply–combine；
3. 股票数量占比与持仓资金暴露为什么不是一回事；
4. 如何在每个交易日的截面上去除行业均值，并识别单成员行业的解释陷阱。

### 输入与预期输出

- 小例子：`2 日 × 4 股票 × 2 行业`的确定收益，用于手工核对；
- 完整行情：`daily_prices.csv` 中 6 只虚构股票的模拟日线；
- 输出：长宽表、行业统计、行业组合收益、持仓行业暴露和中性化诊断；
- 验收：键唯一、标签完整、长宽表往返一致、中性化后每个“日期 × 行业”的均值为 0。

## 本节数据是什么？

本节数据全部离线运行：

- 小例子由代码直接写出，结果完全确定；
- 完整行情是固定随机种子 `365` 生成的**模拟数据**，包含 756 个工作日和 6 只虚构股票；
- 股票代码、行业、价格、成交量和收益不对应任何真实证券。

视频中的十只真实 A 股收益没有随本地材料提供数据来源、复权方式和成分选择规则，因此这里只保留课程方法，不复用其收益数字作为事实结论。

In [1]:
from pathlib import Path
import json
import os
import platform
import sys
import tempfile
import timeit

# Matplotlib 首次运行需要写字体缓存；放到可写临时目录，避免污染用户目录。
MPL_CACHE_DIR = Path(tempfile.gettempdir()) / "pkm-quant-matplotlib"
MPL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPL_CACHE_DIR))

import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd
import seaborn as sns

# 从本机字体中选择支持中文的字体，避免标题显示成方框。
preferred_chinese_fonts = [
    "PingFang SC",
    "Arial Unicode MS",
    "Hiragino Sans GB",
    "Heiti SC",
]
available_font_names = {font.name for font in font_manager.fontManager.ttflist}
chinese_font = next(
    (name for name in preferred_chinese_fonts if name in available_font_names),
    "DejaVu Sans",
)
plt.rcParams["font.sans-serif"] = [chinese_font, "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

def find_project_root():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd / "实验" / "p2-python-toolchain",
        cwd / "p2-python-toolchain",
        *cwd.parents,
    ]
    for candidate in candidates:
        if (candidate / "课程辅助设计.md").exists():
            return candidate
    raise FileNotFoundError("请从 p2-python-toolchain 或量化交易项目目录启动 JupyterLab")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "fixtures"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
RANDOM_SEED = 365

print("Python:", sys.version.split()[0])
print("解释器:", sys.executable)
print("项目:", PROJECT_ROOT)
print("绘图字体:", chinese_font)

Python: 3.11.15
解释器: /opt/homebrew/Caskroom/miniforge/base/envs/pkm-quant/bin/python
项目: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p2-python-toolchain
绘图字体: PingFang SC


## 方法地图

| 方法 | 输入 | 输出 | 本节用途 |
|---|---|---|---|
| `pivot` | 唯一的“日期 × 股票”长表 | 日期为行、股票为列的宽表 | 查看多资产收益矩阵 |
| `stack` | 宽表 | 列标签压入行索引的长 Series | 得到 `date, symbol` MultiIndex |
| `unstack` | MultiIndex Series/DataFrame | 指定索引层展开为列 | 恢复宽表 |
| `groupby().agg()` | 分组后的观测 | 每组一行或多项汇总 | 行业均值、标准差、计数 |
| `groupby().transform()` | 分组后的观测 | 与原数据等长、索引对齐 | 把行业均值放回每一行 |
| `pivot_table` | 可能含重复组合的长表 | 分组聚合后的透视表 | 明确汇总口径 |

`pivot` 不聚合，遇到重复键会报错；`pivot_table` 会聚合，因此必须显式写 `aggfunc`。

## 小例子：先展示原始长表

一行表示某个交易日、某只股票的一期简单收益。两个行业各有两只股票，避免单成员行业的中性化残差被机械地压成 0。

In [2]:
# 构造 2 日 × 4 股票的确定性长表；return 使用小数表示。
mini_panel = pd.DataFrame(
    [
        ("2024-01-02", "ALPHA_1", "行业A", 0.02),
        ("2024-01-02", "ALPHA_2", "行业A", 0.00),
        ("2024-01-02", "BETA_1", "行业B", -0.01),
        ("2024-01-02", "BETA_2", "行业B", 0.01),
        ("2024-01-03", "ALPHA_1", "行业A", 0.01),
        ("2024-01-03", "ALPHA_2", "行业A", 0.03),
        ("2024-01-03", "BETA_1", "行业B", 0.02),
        ("2024-01-03", "BETA_2", "行业B", 0.00),
    ],
    columns=["date", "symbol", "industry", "return"],
)
mini_panel["date"] = pd.to_datetime(mini_panel["date"])

assert mini_panel.shape == (8, 4)
assert not mini_panel.duplicated(["date", "symbol"]).any()
assert mini_panel["industry"].notna().all()
mini_panel

,date,symbol,industry,return
0,2024-01-02,ALPHA_1,行业A,0.02
1,2024-01-02,ALPHA_2,行业A,0.00
2,2024-01-02,BETA_1,行业B,-0.01
3,2024-01-02,BETA_2,行业B,0.01
4,2024-01-03,ALPHA_1,行业A,0.01
5,2024-01-03,ALPHA_2,行业A,0.03
6,2024-01-03,BETA_1,行业B,0.02
7,2024-01-03,BETA_2,行业B,0.00


## `pivot`：把唯一长表变成收益宽表

- `index="date"`：每行一个交易日；
- `columns="symbol"`：每列一只股票；
- `values="return"`：格子里放简单收益率。

由于每个“日期 × 股票”组合只有一行，`pivot` 不需要也不允许静默聚合。

In [3]:
# 将确定性长表重排为 2 × 4 的带标签收益矩阵。
mini_wide = mini_panel.pivot(
    index="date",
    columns="symbol",
    values="return",
).sort_index()

expected_wide = np.array(
    [
        [0.02, 0.00, -0.01, 0.01],
        [0.01, 0.03, 0.02, 0.00],
    ]
)
assert mini_wide.shape == (2, 4)
np.testing.assert_allclose(mini_wide.to_numpy(), expected_wide)
mini_wide

symbol,ALPHA_1,ALPHA_2,BETA_1,BETA_2
date,,,,
2024-01-02,0.02,0.00,-0.01,0.01
2024-01-03,0.01,0.03,0.02,0.00


## `stack` 与 `unstack`：长宽表往返

`stack()` 将股票列压入行索引，得到索引层为 `date, symbol` 的 Series；`unstack("symbol")` 再把股票层展开为列。

有缺失值时，堆叠后的行数可能不是简单的 `日期数 × 股票数`；本例无缺失，因此应有 `2 × 4 = 8` 条观测。重复键、层级名称和排序都要显式检查。

In [4]:
# 宽表堆叠成长 Series，再按 symbol 层恢复原宽表。
mini_stacked = mini_wide.stack().rename("return")
mini_restored = mini_stacked.unstack("symbol")

assert mini_stacked.index.names == ["date", "symbol"]
assert len(mini_stacked) == 8
pd.testing.assert_frame_equal(mini_restored, mini_wide)

display(
    pd.DataFrame(
        {
            "shape": [
                str(mini_wide.shape),
                str(mini_stacked.shape),
                str(mini_restored.shape),
            ]
        },
        index=["原宽表", "stack 后", "unstack 后"],
    )
)
mini_stacked.to_frame()

,shape
原宽表,"(2, 4)"
stack 后,"(8,)"
unstack 后,"(2, 4)"


return
date       symbol         
2024-01-02 ALPHA_1    0.02
           ALPHA_2    0.00
           BETA_1    -0.01
           BETA_2     0.01
2024-01-03 ALPHA_1    0.01
           ALPHA_2    0.03
           BETA_1     0.02
           BETA_2     0.00

## `groupby` 的最小确定性例子

视频用三只股票说明 split–apply–combine。这里使用虚构名称和小数收益：

1. split：按行业标签拆成两组；
2. apply：每组计算算术平均；
3. combine：将两个行业结果拼回带标签 Series。

传入 `groupby` 的标签 Series 会按股票索引对齐；如果标签缺失，默认可能被排除，因此先断言标签完整。

In [5]:
# 两只行业A股票分别涨 2% 和 1%，行业B股票跌 3%。
one_day_move = pd.Series(
    {"ALPHA_1": 0.02, "ALPHA_2": 0.01, "BETA_1": -0.03},
    name="return",
)
one_day_industry = pd.Series(
    {"ALPHA_1": "行业A", "ALPHA_2": "行业A", "BETA_1": "行业B"},
    name="industry",
)

assert one_day_industry.reindex(one_day_move.index).notna().all()
one_day_industry_mean = one_day_move.groupby(one_day_industry).mean()

assert np.isclose(one_day_industry_mean.loc["行业A"], 0.015)
assert np.isclose(one_day_industry_mean.loc["行业B"], -0.03)
one_day_industry_mean.rename("行业平均收益").to_frame()

,行业平均收益
industry,
行业A,0.015
行业B,-0.030


## `agg`：每个行业形成汇总行

`agg` 会压缩行数。下面对小面板的全部“股票日”观测计算均值、样本标准差和数量。

这里的 `sample_std` 混合了日期方向的波动和行业内部股票差异，不能直接称为“行业组合波动率”；真正的行业组合波动要先算每天的行业组合收益，再沿时间求标准差。

In [6]:
# 对每个行业的全部股票日观测执行多项聚合。
mini_industry_agg = mini_panel.groupby("industry")["return"].agg(
    mean_stock_day_return="mean",
    sample_std="std",
    observations="count",
)

assert mini_industry_agg.loc["行业A", "observations"] == 4
assert mini_industry_agg.loc["行业B", "observations"] == 4
mini_industry_agg.round(6)

,mean_stock_day_return,sample_std,observations
industry,,,
行业A,0.015,0.01291,4
行业B,0.005,0.01291,4


## `transform`：逐日去除行业均值

严格的简单行业去均值应在**每个日期的截面**执行：

> 中性化值 = 个股值 − 同日同行业均值

`transform("mean")` 返回与原表等长、索引不变的结果，因此可以安全地逐行相减。这里演示的是简单等权去均值；回归残差法还可以同时控制规模、风格和其他暴露。

In [7]:
# 将同日同行业均值放回每一行，再计算中性化收益。
mini_neutral = mini_panel.copy()
mini_neutral["date_industry_mean"] = mini_neutral.groupby(
    ["date", "industry"]
)["return"].transform("mean")
mini_neutral["neutral_return"] = (
    mini_neutral["return"] - mini_neutral["date_industry_mean"]
)

neutral_check = mini_neutral.groupby(
    ["date", "industry"]
)["neutral_return"].mean()
group_size_check = mini_neutral.groupby(
    ["date", "industry"]
)["symbol"].nunique()

assert mini_neutral["date_industry_mean"].index.equals(mini_panel.index)
assert (group_size_check >= 2).all()
np.testing.assert_allclose(neutral_check.to_numpy(), 0.0, atol=1e-15)
mini_neutral.round(
    {
        "return": 4,
        "date_industry_mean": 4,
        "neutral_return": 4,
    }
)

,date,symbol,industry,return,date_industry_mean,neutral_return
0,2024-01-02,ALPHA_1,行业A,0.02,0.01,0.01
1,2024-01-02,ALPHA_2,行业A,0.00,0.01,-0.01
2,2024-01-02,BETA_1,行业B,-0.01,0.00,-0.01
3,2024-01-02,BETA_2,行业B,0.01,0.00,0.01
4,2024-01-03,ALPHA_1,行业A,0.01,0.02,-0.01
5,2024-01-03,ALPHA_2,行业A,0.03,0.02,0.01
6,2024-01-03,BETA_1,行业B,0.02,0.01,0.01
7,2024-01-03,BETA_2,行业B,0.00,0.01,-0.01


## 股票数量占比不等于资金行业暴露

四只股票在两个行业各有两只，因此按股票数量看是 `50% / 50%`。如果目标权重为 `[50%, 30%, 10%, 10%]`，资金暴露却是 `80% / 20%`。

行业权重 HHI 为各行业资金暴露平方和；本例为 `0.8² + 0.2² = 0.68`。它只是一种集中度诊断，不衡量行业间相关性。

In [8]:
# 用同一组股票对比数量占比、资金暴露和行业权重 HHI。
mini_symbol_industry = (
    mini_panel[["symbol", "industry"]]
    .drop_duplicates()
    .set_index("symbol")["industry"]
    .sort_index()
)
mini_weights = pd.Series(
    {
        "ALPHA_1": 0.50,
        "ALPHA_2": 0.30,
        "BETA_1": 0.10,
        "BETA_2": 0.10,
    },
    name="target_weight",
).sort_index()

count_share = mini_symbol_industry.value_counts(normalize=True)
capital_exposure = mini_weights.groupby(mini_symbol_industry).sum()
mini_exposure_audit = pd.concat(
    [
        count_share.rename("股票数量占比"),
        capital_exposure.rename("资金行业暴露"),
    ],
    axis=1,
).sort_index()
mini_exposure_hhi = float((capital_exposure**2).sum())

assert np.isclose(mini_weights.sum(), 1.0)
assert np.isclose(capital_exposure.sum(), 1.0)
assert np.isclose(mini_exposure_audit.loc["行业A", "股票数量占比"], 0.5)
assert np.isclose(mini_exposure_audit.loc["行业A", "资金行业暴露"], 0.8)
assert np.isclose(mini_exposure_hhi, 0.68)

display(mini_exposure_audit)
pd.DataFrame(
    {"值": [mini_exposure_hhi]},
    index=["行业权重 HHI"],
)

,股票数量占比,资金行业暴露
industry,,
行业A,0.5,0.8
行业B,0.5,0.2


,值
行业权重 HHI,0.68


## 回到完整模拟行情：先看原始长表

`daily_prices.csv` 一行是一个交易日、一只虚构股票的一组 OHLCV 行情。先展示首个日期的全部股票，确认行业标签与价格仍在原始长表中。

In [9]:
# 读取模拟日线，并展示首个日期的 6 只股票。
daily = pd.read_csv(
    DATA_DIR / "daily_prices.csv",
    parse_dates=["date"],
)
first_date = daily["date"].min()
first_date_rows = daily.loc[
    daily["date"].eq(first_date),
    ["date", "symbol", "sector", "adj_close", "volume"],
]

assert daily.shape == (4536, 9)
assert daily["date"].nunique() == 756
assert daily["symbol"].nunique() == 6
first_date_rows

,date,symbol,sector,adj_close,volume
0,2023-01-03,BANK_A,金融,11.7218,2822797
756,2023-01-03,INSURE_A,金融,41.7118,2075343
1512,2023-01-03,EV_A,新能源,27.5575,5113762
2268,2023-01-03,POWER_A,公用事业,8.0817,3733504
3024,2023-01-03,ENERGY_A,能源,6.0296,5425851
3780,2023-01-03,TECH_A,科技,36.2134,6241108


## 构造完整收益宽表

先用 `pivot` 得到带日期和股票标签的复权收盘价矩阵，再按每只股票列计算简单日收益。模拟价格无缺失，因此只删除没有前值的第一日。

In [10]:
# 将原始长表重排为价格宽表，并计算 755 × 6 的日收益宽表。
close = daily.pivot(
    index="date",
    columns="symbol",
    values="adj_close",
).sort_index()
assert close.notna().all().all()

returns_wide = close.pct_change(fill_method=None).iloc[1:]
assert returns_wide.shape == (755, 6)
assert returns_wide.index.is_unique
assert returns_wide.columns.is_unique
returns_wide.head().round(6)

symbol,BANK_A,ENERGY_A,EV_A,INSURE_A,POWER_A,TECH_A
date,,,,,,
2023-01-04,-0.002423,-0.002620,-0.001401,0.006833,0.005927,0.010518
2023-01-05,-0.000599,0.002378,0.010818,-0.005612,0.004108,-0.015522
2023-01-06,-0.003448,0.012193,0.033469,0.021439,0.001482,-0.001207
2023-01-09,-0.013240,-0.012390,-0.012578,-0.019066,0.002128,-0.007248
2023-01-10,-0.000948,0.004182,-0.004978,-0.031666,0.000110,0.004622


## 堆叠收益并对齐行业标签

`stack()` 后一行对应一个“日期 × 股票”收益。行业映射先从原始表去重，并断言每只股票只有一个静态行业标签，再通过股票代码映射到长表。

真实研究不能假设行业永远不变；生产数据应使用带生效日期的 point-in-time 行业映射。

In [11]:
# 构造收益长表，并按 symbol 对齐静态行业标签。
full_symbol_industry = (
    daily[["symbol", "sector"]]
    .drop_duplicates()
    .set_index("symbol")["sector"]
)
assert full_symbol_industry.index.is_unique

returns_long = (
    returns_wide.stack()
    .rename("return")
    .reset_index()
)
returns_long["industry"] = returns_long["symbol"].map(
    full_symbol_industry
)

assert returns_long.shape[0] == 755 * 6
assert not returns_long.duplicated(["date", "symbol"]).any()
assert returns_long["industry"].notna().all()
returns_long.head(8).round({"return": 6})

,date,symbol,return,industry
0,2023-01-04,BANK_A,-0.002423,金融
1,2023-01-04,ENERGY_A,-0.002620,能源
2,2023-01-04,EV_A,-0.001401,新能源
3,2023-01-04,INSURE_A,0.006833,金融
4,2023-01-04,POWER_A,0.005927,公用事业
5,2023-01-04,TECH_A,0.010518,科技
6,2023-01-05,BANK_A,-0.000599,金融
7,2023-01-05,ENERGY_A,0.002378,能源


## `pivot_table`：明确“股票日观测”的聚合口径

下面显式指定 `mean/std/count`。其中标准差来自行业内全部股票日观测，混合了时间变化和股票间差异，因此列名明确写成 `stock_day_std`，不把它冒充行业组合波动率。

In [12]:
# 用 pivot_table 汇总行业内全部股票日观测，并显式指定聚合函数。
industry_pivot = pd.pivot_table(
    returns_long,
    index="industry",
    values="return",
    aggfunc=["mean", "std", "count"],
)
industry_pivot = industry_pivot.droplevel(1, axis=1).rename(
    columns={
        "mean": "mean_stock_day_return",
        "std": "stock_day_std",
        "count": "stock_day_count",
    }
)

assert industry_pivot["stock_day_count"].sum() == len(returns_long)
industry_pivot.round(6)

,mean_stock_day_return,stock_day_std,stock_day_count
industry,,,
公用事业,0.000184,0.008719,755
新能源,0.000614,0.014506,755
科技,0.001154,0.013898,755
能源,0.000124,0.011293,755
金融,0.000445,0.011357,1510


## 正确计算等权行业组合的日收益和波动

先在每个交易日对行业内股票收益取等权均值，得到“日期 × 行业”日收益表；再沿时间计算样本标准差并按 `√252` 年化。

固定成分和每日等权均值隐含每天恢复等权，忽略交易成本和不可交易约束，只用于方法演示。

In [13]:
# 先形成每日等权行业组合，再计算时间序列统计量。
sector_daily_return = returns_long.groupby(
    ["date", "industry"]
)["return"].mean().unstack("industry")

sector_asset_count = full_symbol_industry.value_counts()
sector_statistics = pd.DataFrame(
    {
        "mean_daily_return": sector_daily_return.mean(),
        "daily_volatility": sector_daily_return.std(ddof=1),
        "annualized_volatility": (
            sector_daily_return.std(ddof=1) * np.sqrt(252)
        ),
        "asset_count": sector_asset_count,
    }
)

assert sector_daily_return.shape[0] == returns_wide.shape[0]
assert sector_daily_return.notna().all().all()
sector_statistics.round(6)

,mean_daily_return,daily_volatility,annualized_volatility,asset_count
公用事业,0.000184,0.008719,0.138403,1
新能源,0.000614,0.014506,0.230282,1
科技,0.001154,0.013898,0.220628,1
能源,0.000124,0.011293,0.179270,1
金融,0.000445,0.009989,0.158572,2


### 行业累计收益的口径

下面复利的是“每日等权、每日恢复权重”的行业组合收益：

> 期末累计收益 = `prod(1 + 每日行业组合收益) - 1`

它不同于“先算每只股票两年累计收益，再取行业算术平均”。两种口径都可以描述样本，但不能混称为同一个行业组合。

In [14]:
# 复利每日等权行业收益，并用带标签表格展示期末值。
sector_wealth = (1 + sector_daily_return).cumprod()
sector_terminal_return = sector_wealth.iloc[-1] - 1

assert (sector_wealth > 0).all().all()
sector_terminal_return.rename(
    "每日等权再平衡行业组合期末累计收益"
).to_frame().round(4)

,每日等权再平衡行业组合期末累计收益
industry,
公用事业,0.1165
新能源,0.4683
科技,1.2224
能源,0.0466
金融,0.3479


## 完整行情的行业去均值与单成员陷阱

完整模拟数据中只有金融行业有两只股票，其他行业都只有一只。`transform("mean")` 仍能让每组残差均值为 0，但单成员行业的残差必然为 0，只是因为“自己减自己”。

因此，零残差不能解释为该股票“完全靠行业上涨”，而应标记为没有足够同行可比较。

In [15]:
# 对每个日期、每个行业执行简单等权去均值，并诊断行业样本数。
full_neutral = returns_long.copy()
full_neutral["industry_mean_return"] = full_neutral.groupby(
    ["date", "industry"]
)["return"].transform("mean")
full_neutral["neutral_return"] = (
    full_neutral["return"] - full_neutral["industry_mean_return"]
)

full_neutral_check = full_neutral.groupby(
    ["date", "industry"]
)["neutral_return"].mean()
np.testing.assert_allclose(
    full_neutral_check.to_numpy(),
    0.0,
    atol=1e-15,
)

neutral_diagnostic = pd.DataFrame(
    {
        "asset_count": sector_asset_count,
        "mean_absolute_neutral_return": (
            full_neutral.assign(
                absolute_neutral_return=full_neutral[
                    "neutral_return"
                ].abs()
            )
            .groupby("industry")["absolute_neutral_return"]
            .mean()
        ),
    }
)
neutral_diagnostic["singleton_group"] = (
    neutral_diagnostic["asset_count"] == 1
)
neutral_diagnostic.round(6)

,asset_count,mean_absolute_neutral_return,singleton_group
公用事业,1,0.000000,True
新能源,1,0.000000,True
科技,1,0.000000,True
能源,1,0.000000,True
金融,2,0.004324,False


## 完整模拟持仓的行业暴露

下面给 6 只虚构股票指定一组教学目标权重。行业股票数量占比回答“有几只”，资金暴露回答“投了多少钱”；两者必须分列展示。

In [16]:
# 用确定性目标权重计算资金行业暴露和行业权重 HHI。
target_weights = pd.Series(
    {
        "BANK_A": 0.25,
        "INSURE_A": 0.15,
        "EV_A": 0.30,
        "POWER_A": 0.10,
        "ENERGY_A": 0.10,
        "TECH_A": 0.10,
    },
    name="target_weight",
)
assert set(target_weights.index) == set(full_symbol_industry.index)
assert np.isclose(target_weights.sum(), 1.0)

full_count_share = full_symbol_industry.value_counts(normalize=True)
full_capital_exposure = target_weights.groupby(
    full_symbol_industry
).sum()
full_exposure_audit = pd.concat(
    [
        full_count_share.rename("股票数量占比"),
        full_capital_exposure.rename("资金行业暴露"),
    ],
    axis=1,
).sort_index()
full_exposure_hhi = float((full_capital_exposure**2).sum())

assert np.isclose(full_capital_exposure.sum(), 1.0)
assert np.isclose(full_exposure_hhi, 0.28)
display(full_exposure_audit.round(4))
pd.DataFrame(
    {"值": [full_exposure_hhi]},
    index=["行业权重 HHI"],
)

,股票数量占比,资金行业暴露
sector,,
公用事业,0.1667,0.1
新能源,0.1667,0.3
科技,0.1667,0.1
能源,0.1667,0.1
金融,0.3333,0.4


,值
行业权重 HHI,0.28


## 可复用结论

1. 先用 pandas 标签保证日期、股票和行业对齐，再做分组统计；
2. `agg` 用于汇总，`transform` 用于把组统计量放回原行；
3. `stack/unstack` 前后都要检查层级名、唯一键、缺失组合和 shape；
4. 行业数量集中度、资金暴露和收益相关性是不同诊断；
5. 中性化应按每个历史时点执行，并保存中性化前后分布和暴露检查。

## 方法边界

- 视频的真实股票收益缺少可复现来源，本 Notebook 不据此作市场结论；
- 当前行业分类不能回填历史，必须使用带生效日期的行业映射；
- `groupby` 默认可能排除缺失行业标签，计算前必须断言标签完整；
- 单成员和小样本行业无法稳定估计同行基准；
- 等权行业均值不等于市值加权指数；
- 简单去行业均值不会自动消除规模、价值、流动性等其他风格暴露；
- 若策略目标就是行业轮动，中性化可能删除目标信号；
- 全样本统计是描述性结果，不能作为同一期间的无未来信息选股证据。

## 练习

1. 把 `mini_weights` 改为四只股票等权，比较数量占比、资金暴露和 HHI。
2. 删除 `mini_panel` 的一条记录，观察 `stack` 行数和 `unstack` 后的缺失位置。
3. 给小例子增加第三个只有一只股票的行业，解释为什么其中性化残差恒为 0。
4. 将行业均值改为持仓权重加权均值，并验证每组的加权残差均值为 0。